# Dimension for CUSTOMER Context

# View Schema of Tables being used

In [0]:
df1 = spark.table("databricks_medallion_pipeline.silver.crm_customers")
print("Schema of CRM_Customers")
df1.printSchema()

df2 = spark.table("databricks_medallion_pipeline.silver.erp_customers")
print("Schema of ERP_Customers")
df2.printSchema()

df3 = spark.table("databricks_medallion_pipeline.silver.erp_customer_location")
print("Schema of ERP_Customers_Location")
df3.printSchema()

# Transformation Logic

In [0]:
query = """
SELECT
    ROW_NUMBER() OVER (ORDER BY cc.customer_id) AS customer_key,
    cc.customer_id,
    cc.customer_number,
    cc.first_name,
    cc.last_name,
    ecl.country,
    cc.marital_status,
    CASE 
        WHEN cc.gender <> 'n/a' THEN cc.gender
        ELSE COALESCE(ec.gender, 'n/a')
    END as gender,
    ec.birth_date AS birthdate,
    cc.created_date AS create_date
FROM databricks_medallion_pipeline.silver.crm_customers cc
LEFT JOIN databricks_medallion_pipeline.silver.erp_customers ec
    ON cc.customer_number = ec.customer_number
LEFT JOIN databricks_medallion_pipeline.silver.erp_customer_location ecl
    ON cc.customer_number = ecl.customer_number
"""

df = spark.sql(query)

In [0]:
df.printSchema()
df.limit(10).display()

#Writing the Gold Table

In [0]:
df.write.mode("overwrite").saveAsTable("databricks_medallion_pipeline.gold.dim_customers")

# Verification of Gold Table

In [0]:
%sql
SELECT * FROM databricks_medallion_pipeline.gold.dim_customers LIMIT 10